In [22]:
import pandas as pd
import numpy as np
import os

In [2]:
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)

injury_path = RAW_PATH + "InjuryRecord.csv"
playlist_path = RAW_PATH + "PlayList.csv"
tracking_path = RAW_PATH + "PlayerTrackData.csv"

In [3]:
injury = pd.read_csv(injury_path)
playlist = pd.read_csv(playlist_path)

print("Injury:", injury.shape)
print("Playlist:", playlist.shape)

Injury: (105, 9)
Playlist: (267005, 14)


In [4]:
key_columns = ["PlayKey", "GameID"]

for col in key_columns:
    injury[col] = injury[col].astype("string").str.strip()
    playlist[col] = playlist[col].astype("string").str.strip()

injury["PlayerKey"] = pd.to_numeric(
    injury["PlayerKey"], errors="coerce"
)

playlist["PlayerKey"] = pd.to_numeric(
    playlist["PlayerKey"], errors="coerce"
)

print("Identifier types standardized.")

Identifier types standardized.


In [5]:
print("Injury duplicate rows:",
      injury.duplicated().sum())

print("Playlist duplicate rows:",
      playlist.duplicated().sum())

print("\nDuplicate Injury PlayKeys:",
      injury["PlayKey"].dropna().duplicated().sum())

print("Duplicate Playlist PlayKeys:",
      playlist["PlayKey"].dropna().duplicated().sum())

Injury duplicate rows: 0
Playlist duplicate rows: 0

Duplicate Injury PlayKeys: 1
Duplicate Playlist PlayKeys: 0


In [6]:
print(
    "Playlist unique PlayKeys:",
    playlist["PlayKey"].nunique()
)

print(
    "Playlist total non-null PlayKeys:",
    playlist["PlayKey"].notna().sum()
)

Playlist unique PlayKeys: 267005
Playlist total non-null PlayKeys: 267005


In [7]:
missing_injury_playkey = injury[
    injury["PlayKey"].isna()
]

available_injury_playkey = injury[
    injury["PlayKey"].notna()
]

print("Injury records with PlayKey:",
      len(available_injury_playkey))

print("Injury records without PlayKey:",
      len(missing_injury_playkey))

display(missing_injury_playkey.head(10))

Injury records with PlayKey: 77
Injury records without PlayKey: 28


,PlayerKey,GameID,PlayKey,BodyPart,Surface,DM_M1,DM_M7,DM_M28,DM_M42
46,33337,33337-2,<NA>,Foot,Natural,1,1,1,1
47,45099,45099-5,<NA>,Knee,Natural,1,1,1,1
48,36591,36591-9,<NA>,Knee,Natural,1,1,1,1
49,45950,45950-6,<NA>,Toes,Synthetic,1,1,0,0
50,39653,39653-4,<NA>,Ankle,Synthetic,1,0,0,0
51,38253,38253-10,<NA>,Toes,Synthetic,1,1,1,0
52,38214,38214-12,<NA>,Toes,Synthetic,1,0,0,0
53,43119,43119-12,<NA>,Ankle,Synthetic,1,0,0,0
54,35648,35648-12,<NA>,Ankle,Natural,1,0,0,0
55,40051,40051-13,<NA>,Ankle,Natural,1,0,0,0


In [12]:
injury_key_check = available_injury_playkey[
    ["PlayKey"]
].merge(
    playlist[["PlayKey"]],
    on="PlayKey",
    how="left",
    indicator=True,
    validate="many_to_one"
)

print(injury_key_check["_merge"].value_counts())

_merge
both          77
left_only      0
right_only     0
Name: count, dtype: int64


In [14]:
# Check whether PlayKey is unique in the playlist
print("Duplicate PlayKeys in playlist:",
      playlist["PlayKey"].duplicated().sum())

# Stop if playlist keys are not unique
assert playlist["PlayKey"].is_unique, (
    "Playlist contains duplicate PlayKeys. "
    "Investigate these before merging."
)

# Merge injury records with playlist metadata
injury_playlist_check = available_injury_playkey.merge(
    playlist[["PlayKey", "PlayerKey", "GameID"]],
    on="PlayKey",
    how="left",
    suffixes=("_injury", "_playlist"),
    indicator=True,
    validate="many_to_one"
)

display(injury_playlist_check.head())

Duplicate PlayKeys in playlist: 0


,PlayerKey_injury,GameID_injury,PlayKey,BodyPart,Surface,DM_M1,DM_M7,DM_M28,DM_M42,PlayerKey_playlist,GameID_playlist,_merge
0,39873,39873-4,39873-4-32,Knee,Synthetic,1,1,1,1,39873,39873-4,both
1,46074,46074-7,46074-7-26,Knee,Natural,1,1,0,0,46074,46074-7,both
2,36557,36557-1,36557-1-70,Ankle,Synthetic,1,1,1,1,36557,36557-1,both
3,46646,46646-3,46646-3-30,Ankle,Natural,1,0,0,0,46646,46646-3,both
4,43532,43532-5,43532-5-69,Ankle,Synthetic,1,1,1,1,43532,43532-5,both


In [15]:
print("Merge results:")
print(injury_playlist_check["_merge"].value_counts())

matched_records = injury_playlist_check[
    injury_playlist_check["_merge"] == "both"
]

unmatched_records = injury_playlist_check[
    injury_playlist_check["_merge"] == "left_only"
]

print("Matched injury records:", len(matched_records))
print("Unmatched injury records:", len(unmatched_records))

Merge results:
_merge
both          77
left_only      0
right_only     0
Name: count, dtype: int64
Matched injury records: 77
Unmatched injury records: 0


In [16]:
# Check PlayerKey mismatches
player_mismatches = injury_playlist_check[
    injury_playlist_check["PlayerKey_injury"] !=
    injury_playlist_check["PlayerKey_playlist"]
]

# Check GameID mismatches
game_mismatches = injury_playlist_check[
    injury_playlist_check["GameID_injury"] !=
    injury_playlist_check["GameID_playlist"]
]

print("PlayerKey mismatches:", len(player_mismatches))
print("GameID mismatches:", len(game_mismatches))

display(player_mismatches.head())
display(game_mismatches.head())

PlayerKey mismatches: 0
GameID mismatches: 0


,PlayerKey_injury,GameID_injury,PlayKey,BodyPart,Surface,DM_M1,DM_M7,DM_M28,DM_M42,PlayerKey_playlist,GameID_playlist,_merge


,PlayerKey_injury,GameID_injury,PlayKey,BodyPart,Surface,DM_M1,DM_M7,DM_M28,DM_M42,PlayerKey_playlist,GameID_playlist,_merge


In [17]:
tracking_playkeys = set()

for chunk in pd.read_csv(
    tracking_path,
    usecols=["PlayKey"],
    dtype={"PlayKey": "string"},
    chunksize=1_000_000
):
    tracking_playkeys.update(
        chunk["PlayKey"].dropna().unique()
    )

print("Unique tracking PlayKeys:", len(tracking_playkeys))

Unique tracking PlayKeys: 266960


In [18]:
playlist_keys = set(playlist["PlayKey"].dropna())

common_keys = playlist_keys.intersection(tracking_playkeys)

playlist_only = playlist_keys - tracking_playkeys

tracking_only = tracking_playkeys - playlist_keys

print("Playlist unique keys:", len(playlist_keys))
print("Tracking unique keys:", len(tracking_playkeys))
print("Common keys:", len(common_keys))
print("Keys only in playlist:", len(playlist_only))
print("Keys only in tracking:", len(tracking_only))

Playlist unique keys: 267005
Tracking unique keys: 266960
Common keys: 266960
Keys only in playlist: 45
Keys only in tracking: 0


In [19]:
# Merge all available injury records with playlist metadata
injury_metadata = available_injury_playkey.merge(
    playlist,
    on="PlayKey",
    how="left",
    suffixes=("_injury", "_playlist"),
    indicator=True,
    validate="many_to_one"
)

print("Final injury metadata shape:", injury_metadata.shape)

display(injury_metadata.head())

Final injury metadata shape: (77, 23)


,PlayerKey_injury,GameID_injury,PlayKey,BodyPart,Surface,DM_M1,DM_M7,DM_M28,DM_M42,PlayerKey_playlist,...,PlayerGame,StadiumType,FieldType,Temperature,Weather,PlayType,PlayerGamePlay,Position,PositionGroup,_merge
0,39873,39873-4,39873-4-32,Knee,Synthetic,1,1,1,1,39873,...,4,Indoors,Synthetic,84,Cloudy,Punt,32,OLB,LB,both
1,46074,46074-7,46074-7-26,Knee,Natural,1,1,0,0,46074,...,7,Open,Natural,76,Partly Cloudy,Punt,26,OLB,LB,both
2,36557,36557-1,36557-1-70,Ankle,Synthetic,1,1,1,1,36557,...,1,Outdoor,Synthetic,63,Clear and warm,Pass,70,SS,DB,both
3,46646,46646-3,46646-3-30,Ankle,Natural,1,0,0,0,46646,...,3,Outdoor,Natural,80,Cloudy,Punt,30,LB,LB,both
4,43532,43532-5,43532-5-69,Ankle,Synthetic,1,1,1,1,43532,...,5,Retractable Roof,Synthetic,89,Partly Cloudy,Kickoff,69,WR,WR,both


In [20]:
# Save merged injury metadata
injury_metadata.to_csv(
    PROCESSED_PATH + "injury_play_metadata.csv",
    index=False
)

# Save injury records without PlayKey separately
missing_injury_playkey.to_csv(
    PROCESSED_PATH + "unmatched_injury_records.csv",
    index=False
)

print("Datasets saved successfully!")

Datasets saved successfully!


In [21]:
# Reload saved dataset
final_data = pd.read_csv(
    PROCESSED_PATH + "injury_play_metadata.csv"
)

print("Final dataset shape:", final_data.shape)

print("\nSurface distribution:")
print(final_data["FieldType"].value_counts(dropna=False))

print("\nBody part distribution:")
print(final_data["BodyPart"].value_counts(dropna=False))

print("\nMissing values:")
display(final_data.isnull().sum())

Final dataset shape: (77, 23)

Surface distribution:
FieldType
Synthetic    41
Natural      36
Name: count, dtype: int64

Body part distribution:
BodyPart
Knee     36
Ankle    35
Foot      6
Name: count, dtype: int64

Missing values:


PlayerKey_injury      0
GameID_injury         0
PlayKey               0
BodyPart              0
Surface               0
DM_M1                 0
DM_M7                 0
DM_M28                0
DM_M42                0
PlayerKey_playlist    0
GameID_playlist       0
RosterPosition        0
PlayerDay             0
PlayerGame            0
StadiumType           4
FieldType             0
Temperature           0
Weather               3
PlayType              0
PlayerGamePlay        0
Position              0
PositionGroup         0
_merge                0
dtype: int64